In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api
# for integration step
!pip install pyngrok fastapi uvicorn nest_asyncio

# Libraries

from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from transformers import AutoTokenizer

import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

import json
from google.colab import userdata

# Path to claims JSON file (persists across sessions)
claims_file = "/content/drive/MyDrive/Claims/video_claims.json"

os.makedirs(os.path.dirname(claims_file), exist_ok=True)

#---------------------------------
# Run claim extraction pipeline if no JSON exists
#---------------------------------

if not os.path.exists(claims_file):

  #---------------------------------
  # Extract videos data using youtube api
  #---------------------------------

  api_key = userdata.get("API_KEY")

  yta = YouTubeTranscriptApi()

  youtube = build('youtube', 'v3', developerKey=api_key)

  # Get top 50 game video (video id & title)

  request = youtube.videos().list(
      part='snippet',
      chart='mostPopular',
      regionCode='us',
      videoCategoryId='20',
      maxResults='5'
  )

  response = request.execute()

  videos_data = []

  for item in response["items"]:
      video_id = item.get("id")

      title = item.get("snippet").get("title")

      videos_data.append({"Video ID": video_id, "Title": title})

  df = pd.DataFrame(videos_data)

  #---------------------------------
  # Load Qwen model
  #---------------------------------

  # Load tokenizer with left padding for Qwen
  tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side ='left')

  pipe = pipeline(
      "text-generation",
      model="Qwen/Qwen2.5-1.5B-Instruct",
      tokenizer = tokenizer,
      torch_dtype="auto",
      device_map="auto",
      max_new_tokens=200 # increase to prevent incomplete output
  )

  #---------------------------------
  # Clean raw transcript
  #---------------------------------

  def clean_transcript(transcript):
    cleaned = [] # list for cleaned snippets

    # Loop to get text (clean) & get start time
    for snippet in transcript.snippets:
      text = snippet.text # get text from snippet
      start_time = snippet.start # get start time from snippet

      while "[" in text:
        start = text.index("[")
        end = text.index("]") + 1
        caption = text[start:end]
        text = text.replace(caption, "")

      cleaned.append({"text": text, "start": start_time}) # add snippet's text and start time to cleaned list

    return cleaned

  #---------------------------------
  # Chunk cleaned transcript
  #---------------------------------

  def chunk_transcript(cleaned):
    chunk_size = 300 # chunk size based on time (sec)
    chunks = [] # list for chunks
    chunk_snippets = [] # list to build each chunk
    curr_chunk_time = 0

    # Loop through all snippets in cleaned[]
    for snippet in cleaned:
        text = snippet["text"] # get snippet text
        chunk_snippets.append(text) # add text to chunk_snippets list

        # Check if reach chunk_size (approx.)
        if snippet["start"] - curr_chunk_time >= chunk_size:
          chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
          chunks.append(chunk) # add chunk to list of chunks

          chunk_snippets = [] # reset chunk_snippets list
          curr_chunk_time = snippet["start"] # reset time to current start time

    # If leftover snippets, create and add last chunk
    if chunk_snippets:
      chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
      chunks.append(chunk) # add chunk to list of chunks

    return chunks

  #---------------------------------
  # Claim extraction
  #---------------------------------

  def claim_extraction(chunks):
    message_list = []
    # Create message more each chunk
    for chunk in chunks:
      messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
        {"role": "user", "content": f"Extract one concise high-level claim that summarizes the main point.\n\nText: {chunk}"}
      ]
      message_list.append(messages) # add message to list

    results = []
    # Send messages to Qwen model (process in batches)
    for output in pipe(message_list, batch_size=5, max_length=None, temperature=0.0, do_sample=False):
      # Get claim from Qwen response
      result = output[0]["generated_text"][-1]["content"]
      results.append(result) # add qwen output to list
    return results

  #---------------------------------
  # Run claim extraction pipeline
  #---------------------------------

  # test all 50 videos
  video_ids = df["Video ID"].tolist() # video id
  video_titles = df["Title"].tolist() # video title
  video_claims = []

  # Loop through each video id in list
  for i in range(len(video_ids)):
    video_id = video_ids[i]
    title = video_titles[i]
    try:
      transcript = yta.fetch(video_id) # get transcript

      # Filter videos based on time (keep only <=30 minutes)
      video_time = transcript.snippets[-1].start + transcript.snippets[-1].duration
      if video_time <= 1800:

        chunks = chunk_transcript(clean_transcript(transcript)) # clean/chunk transcript

        # Extract claims from chunks list
        claims_list = claim_extraction(chunks)

      else:
        print(f"Video {video_id} is too long. Video time: {video_time}")

        video_claims.append({"video_id": video_id, "title": title, "claims": claims_list})
    except Exception as e:
      print(f"Error for {video_id}: {type(e).__name__}: {e}")

  with open(claims_file, 'w') as file:
    json.dump(video_claims, file, indent=4)

else:
  print("JSON file already exists")

#---------------------------------
# Pipeline Integration Step: FastAPI
#---------------------------------

from fastapi import FastAPI, HTTPException # for FastAPI app and handle HTTP requests
from fastapi.middleware.cors import CORSMiddleware # for resource sharing
import uvicorn # for server that runs FastAPI app
from pyngrok import ngrok # to expose FastAPI on public internet
import nest_asyncio # to run FastAPI on Colab

# apply nest_asyncio for run async code in Colab
nest_asyncio.apply()

# Creat FastAPI app
app = FastAPI(title="Claims API")

# Endpoint to serve claims
@app.get("/")
async def get_claims():
  try:
    # try load claims from json file
    with open(claims_file, "r") as file:
      video_claims = json.load(file)
    return video_claims
  except FileNotFoundError:
    # if file not found
    raise HTTPException(status_code=404, detail="Claims file not found")

# Launch FastAPI app

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

# for run in Colab, expose app via ngrok
public_url = ngrok.connect(8000) # port 8000 for FastAPI
print(f"Public URL:", public_url) # public URL for access

# Run FastAPI app using Uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


